# Pré-processamento 1 — Maceió

Este notebook realiza:
1. Carregamento do CSV filtrado
2. Obtenção do polígono municipal via OpenStreetMap
3. Cálculo dinâmico do tamanho do grid
4. Geração da máscara de validade (células dentro do polígono)
5. Filtragem espacial dos pontos de crime
6. Visualização em mapa interativo

**Entrada:** `./output/maceio/filtered_maceio.csv`

**Saída:** `osm_maceio.geojson`, `maceio_mask_final.npy`, `filtered_maceio_inside_polygon.csv`

## 1. Carregamento dos dados filtrados de Maceió

In [1]:
# Carrega o CSV de Maceió gerado pelo notebook pre-processing.ipynb

import pandas as pd

df_maceio = pd.read_csv(
    "./output/maceio/filtered_maceio.csv", delimiter=";", low_memory=False
)

In [2]:
display(df_maceio)

,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO
0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió
1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió
2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió
3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió
4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió
...,...,...,...,...,...
72719,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió
72720,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió
72721,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió
72722,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió


## 2. Obtenção do polígono municipal via OpenStreetMap

Consulta o OpenStreetMap para obter o polígono dos limites de Maceió. Seleciona o maior polígono encontrado e salva em GeoJSON.

In [3]:
# Busca o polígono de Maceió no OpenStreetMap
# Seleciona o maior polígono (por área) entre os resultados
# Salva em formato GeoJSON para uso nas etapas seguintes

import osmnx as ox
import geopandas as gpd

place = "Maceió, Alagoas, Brazil"
print(f"Querying OSM for: {place}")

gdf_place = ox.geocode_to_gdf(place)
polys = gdf_place[gdf_place.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    # fallback: fetch administrative boundaries and pick candidates
    tags = {"boundary": "administrative"}
    adm = ox.geometries_from_place(place, tags=tags)
    polys = adm[adm.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    raise SystemExit(
        "Could not find polygon for Maceió on OSM. Check query or network."
    )

# # compute area in metric projection to pick the largest polygon
polys = polys.to_crs("EPSG:3857")
polys["area_m2"] = polys.geometry.area
largest_idx = polys["area_m2"].idxmax()
poly = polys.loc[largest_idx].geometry
# convert back to WGS84
poly = gpd.GeoSeries([poly], crs="EPSG:3857").to_crs("EPSG:4326").iloc[0]

osm_maceio = gpd.GeoDataFrame({"name": ["Maceió"], "geometry": [poly]}, crs="EPSG:4326")
osm_path = "./output/maceio/osm_maceio.geojson"
osm_maceio.to_file(osm_path, driver="GeoJSON")

Querying OSM for: Maceió, Alagoas, Brazil


## 3. Cálculo do tamanho do grid

Calcula N a partir da área do bounding box, visando células de ~0,2 km².

#### Grid size

Following Albors Zumel et al. (2025), each cell should cover ~0.077 sq. miles (~0.2 km²). `GRID_SIZE` is derived from the city's bounding box so the same block works for any city whose polygon is stored in `osm_maceio.geojson`.

In [4]:
import math
import geopandas as gpd

TARGET_CELL_KM2 = 0.2  # 0.077 sq. miles

_mun = gpd.read_file("./output/maceio/osm_maceio.geojson").to_crs("EPSG:4326")
_gdf = _mun[_mun["name"] == "Maceió"]
_min_lon, _min_lat, _max_lon, _max_lat = _gdf.total_bounds
_lat_c = (_min_lat + _max_lat) / 2
_bbox_km2 = (
    (_max_lon - _min_lon) * 111.320 * math.cos(math.radians(_lat_c))
    * (_max_lat - _min_lat) * 110.574
)
GRID_SIZE = round(math.sqrt(_bbox_km2 / TARGET_CELL_KM2))
print(f"Maceió bbox: {_bbox_km2:.1f} km^2")
print(f"GRID_SIZE = {GRID_SIZE}  ({GRID_SIZE**2} cells of ~{TARGET_CELL_KM2} km^2)")

Maceió bbox: 1063.3 km^2
GRID_SIZE = 73  (5329 cells of ~0.2 km^2)


## 4. Geração da máscara de validade

Cria uma matriz binária N×N indicando quais células do grid estão dentro do polígono municipal.

In [5]:
# Gera a máscara binária de validade
# Para cada célula do grid, verifica se há interseção com o polígono municipal
# 1 = célula válida (dentro da cidade), 0 = fora

import numpy as np
import geopandas as gpd
import shapely.geometry

grid_size = GRID_SIZE

mun = gpd.read_file("./output/maceio/osm_maceio.geojson")
mun = mun.to_crs("EPSG:4326")
gdf = mun[mun["name"] == "Maceió"].copy()
city_boundary = gdf.union_all()

bbox = gdf.total_bounds
min_lon, min_lat, max_lon, max_lat = bbox[2], bbox[1], bbox[0], bbox[3]

lon = np.linspace(min_lon, max_lon, grid_size + 1)
lat = np.linspace(min_lat, max_lat, grid_size + 1)

mask = np.zeros((grid_size, grid_size), dtype=np.float32)

for i in range(grid_size):
    for j in range(grid_size):
        cell_box = shapely.geometry.box(lon[j], lat[i], lon[j + 1], lat[i + 1])
        if city_boundary.intersects(cell_box):
            mask[i, j] = 1

np.save("./output/maceio/maceio_mask_final.npy", mask)
print(f"Mask shape: {mask.shape}, valid cells: {int(mask.sum())}/{grid_size**2}")

Mask shape: (73, 73), valid cells: 2714/5329


## 5. Filtragem espacial dos pontos de crime

Realiza um spatial join entre os pontos de ocorrência e o polígono municipal, mantendo apenas os registros dentro dos limites de Maceió.

In [6]:
# Spatial join: mantém apenas pontos dentro do polígono municipal
# Remove registros fora dos limites da cidade

# Spatial join points with municipality polygons (geojs-27-mun.json)
# Requires: geopandas
import geopandas as gpd

# load municipalities and ensure CRS is EPSG:4326
mun = gpd.read_file("./output/maceio/osm_maceio.geojson")
mun = mun.to_crs("EPSG:4326")

# create GeoDataFrame for Maceió points
gdf_maceio = gpd.GeoDataFrame(
    df_maceio.copy(),
    geometry=gpd.points_from_xy(df_maceio["LONGITUDE"], df_maceio["LATITUDE"]),
    crs="EPSG:4326",
)

# spatial join: attach municipality 'name' to each point (left join)
# use predicate='within' so points must fall inside polygon
joined = gpd.sjoin(
    gdf_maceio, mun[["name", "geometry"]], how="left", predicate="within"
)

# boolean: specifically in Maceió municipality
joined["in_maceio"] = joined["name"] == "Maceió"

# summary counts
total = len(df_maceio)
inside_count = int(joined["in_maceio"].sum())
outside_count = total - inside_count
print(f"Maceió records total: {total}")
print(f" - inside Maceió polygon: {inside_count}")
print(f" - outside Maceió polygon: {outside_count}")

# Filter df_maceio to only records whose points lie inside the Maceió polygon
inside_maceio = joined.loc[joined["in_maceio"]].copy()

inside_maceio_clean = inside_maceio.drop(
    columns=["geometry", "name", "in_maceio", "index_right"]
)

# reset index and overwrite df_maceio with filtered rows
df_maceio = inside_maceio_clean.reset_index(drop=True)

# save filtered CSV
output_path = "./output/maceio/filtered_maceio_inside_polygon.csv"
df_maceio.to_csv(output_path, index=False, sep=";")
print(f"Saved {len(df_maceio)} Maceió rows inside polygon to {output_path}")

Maceió records total: 72724
 - inside Maceió polygon: 72649
 - outside Maceió polygon: 75
Saved 72649 Maceió rows inside polygon to ./output/maceio/filtered_maceio_inside_polygon.csv


## 6. Visualização em mapa interativo

Exibe o polígono municipal e os pontos de crime (dentro/fora) em um mapa folium.

In [7]:
# Visualização: mapa interativo com polígono e pontos de crime

import folium

# Show the map with the polygon and the points inside/outside the polygon
osm = gpd.read_file("./output/maceio/osm_maceio.geojson").to_crs("EPSG:4326")
centroid = osm.geometry.centroid.iloc[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)

folium.GeoJson(
    osm.to_json(),
    name="Maceió (OSM)",
    style_function=lambda f: {
        "fillColor": "transparent",
        "color": "blue",
        "weight": 2,
        "opacity": 0.8,
    },
).add_to(m)

# optional: overlay sample points if gdf_maceio exists
if "gdf_maceio" in globals():
    for _, r in joined.iterrows():
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x],
            radius=2,
            color="green" if r["in_maceio"] else "red",
            fill=True,
        ).add_to(m)
m.save("./output/maceio/maceio_and_points.html")
print("Saved maceio_osm_polygon_map.html")

/tmp/ipykernel_3141894/196105593.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = osm.geometry.centroid.iloc[0]


Saved maceio_osm_polygon_map.html
